# Метод опорных векторов: разделение классов с максимальным зазором

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Метод опорных векторов».

## Что делает метод

Метод опорных векторов (SVM) строит разделяющую границу между классами так, чтобы зазор до ближайших наблюдений был максимальным. Решение определяется лишь немногими граничными наблюдениями — опорными векторами. Ядровый приём позволяет строить нелинейные границы, не вычисляя признаки в пространстве высокой размерности явно.

Метод применим к задачам классификации по таблице измерений, особенно когда классы разделимы сложной, нелинейной границей. В этом блокноте мы обучим SVM с разными ядрами, подберём параметры и визуализируем разделяющую границу. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вам нужно разделить два вида клеток на предметном стекле: красные и синие. Вы рисуете границу между ними. Но какую именно? Граница, которая проходит максимально далеко от ближайших точек обоих классов, — самая «безопасная»: при небольших измерительных ошибках она не будет нарушена. Именно эту идею реализует метод опорных векторов: он ищет разделяющую границу с максимальным «зазором» до ближайших наблюдений.

Наблюдения, которые оказались ближайшими к границе и фактически её определяют, называются **опорными векторами**. Важно: на решение влияют только они, все остальные точки роли не играют — это делает модель компактной и устойчивой.

Что делать, если классы нельзя разделить прямой линией? Здесь помогает **ядровый приём**: данные неявно «переносятся» в пространство более высокой размерности, где они становятся линейно разделимы. RBF-ядро, например, позволяет строить произвольно сложные нелинейные границы.

**Ключевые термины:**
- **Зазор (margin)** — расстояние между разделяющей границей и ближайшими к ней точками; SVM максимизирует зазор.
- **Опорные векторы** — наблюдения, лежащие ближайшими к границе и определяющие её положение.
- **Ядро (kernel)** — функция, неявно задающая пространство признаков высокой размерности; RBF-ядро создаёт нелинейную границу.
- **Параметр C** — управляет «строгостью»: малое C допускает нарушения зазора ради его ширины (мягкая граница); большое C требует правильной классификации всех точек (жёсткая граница, риск переобучения).
- **Параметр gamma** (для RBF) — управляет «радиусом влияния» каждой точки; большое gamma — узкое влияние, извилистая граница.
- **GridSearchCV** — перебор всех комбинаций гиперпараметров с кросс-валидацией для нахождения лучшего сочетания.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем два набора. Для наглядной визуализации границы — синтетический двумерный набор `make_moons` с двумя переплетёнными классами. Для оценки качества на реальных данных — открытый набор `breast_cancer` из `scikit-learn` (569 наблюдений, 30 признаков).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons, load_breast_cancer

# двумерный набор для визуализации границы
X2d, y2d = make_moons(n_samples=300, noise=0.25, random_state=42)

# реальный набор для оценки качества
real = load_breast_cancer(as_frame=True)
X = real.data
y = real.target
class_names = list(real.target_names)

print(f'Двумерный набор: {X2d.shape[0]} наблюдений, 2 признака')
print(f'Реальный набор: {X.shape[0]} наблюдений, {X.shape[1]} признаков')

### Наглядный «ага»-эксперимент: зазор, граница и опорные векторы на линейном примере

До перехода к сложным ядрам, посмотрим на главную идею SVM — максимальный зазор — на простом линейно разделимом примере. На графике ниже: пунктирные линии — это «стенки зазора», сплошная — разделяющая граница по центру. Обведённые точки — опорные векторы.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

rng_svm = np.random.default_rng(12)
n_lin = 40
# Линейно разделимые данные
X_lin = np.vstack([
    rng_svm.multivariate_normal([-1.5, 0], [[0.4, 0], [0, 0.4]], n_lin),
    rng_svm.multivariate_normal([1.5, 0], [[0.4, 0], [0, 0.4]], n_lin),
])
y_lin = np.array([0] * n_lin + [1] * n_lin)

svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_lin, y_lin)

# Граница решения и зазор
w = svm_lin.coef_[0]
b = svm_lin.intercept_[0]
x_range = np.linspace(X_lin[:, 0].min() - 0.5, X_lin[:, 0].max() + 0.5, 100)
# Граница: w[0]*x + w[1]*y + b = 0  =>  y = -(w[0]*x + b) / w[1]
y_boundary = -(w[0] * x_range + b) / w[1]
y_margin_up = -(w[0] * x_range + b - 1) / w[1]
y_margin_dn = -(w[0] * x_range + b + 1) / w[1]

fig, ax = plt.subplots(figsize=(9.5, 6.0))

# Зазор (заливка)
ax.fill_between(x_range, y_margin_dn, y_margin_up,
                alpha=0.12, color=VIZ['series'][1], label='зазор (margin)')

# Точки
for cls, color, label in zip(
        [0, 1],
        [VIZ['series'][0], VIZ['series'][2]],
        ['класс 0', 'класс 1']):
    m = y_lin == cls
    ax.scatter(X_lin[m, 0], X_lin[m, 1],
               color=color, edgecolor='white', s=50, zorder=3,
               label=label)

# Опорные векторы
sv_idx = svm_lin.support_
ax.scatter(X_lin[sv_idx, 0], X_lin[sv_idx, 1],
           s=200, facecolors='none',
           edgecolors=VIZ['series'][3], linewidths=2.2,
           label='опорные векторы', zorder=4)

# Границы
ax.plot(x_range, y_boundary,
        color=VIZ['series'][3], linewidth=2.5, label='граница решения')
ax.plot(x_range, y_margin_up,
        color=VIZ['series'][1], linewidth=1.5, linestyle='--', label='граница зазора')
ax.plot(x_range, y_margin_dn,
        color=VIZ['series'][1], linewidth=1.5, linestyle='--')

ax.set_xlabel('Признак 1')
ax.set_ylabel('Признак 2')
ax.set_title('SVM: граница решения, зазор и опорные векторы')
ax.legend(loc='upper left', fontsize=10)
ax.grid(False)
fig.tight_layout()
plt.show()

print(f'Число опорных векторов: {len(sv_idx)} из {len(y_lin)} наблюдений')

**Как читать этот график.** Сплошная тёмная линия — разделяющая граница. Зелёная заливка между двумя пунктирными линиями — это зазор: SVM максимизирует его ширину. Обведённые кружками точки — опорные векторы: именно они лежат на границах зазора и определяют всю геометрию решения. Обратите внимание: остальные 70+ точек вообще не участвуют в формировании границы. Это делает SVM устойчивым к «далёким» наблюдениям, но чувствительным к опорным векторам.

## 4. Применение метода

Обучим SVM на реальном наборе (рак молочной железы, 30 признаков). Признаки обязательно стандартизуются: метод чувствителен к масштабу, поскольку опирается на расстояния. Для подбора гиперпараметров используем **GridSearchCV** — он перебирает все комбинации заданных значений C и gamma и выбирает лучшую по кросс-валидации.

**Параметр C** (от «Cost» — цена): малое C допускает нарушения зазора, строит широкий зазор с некоторыми ошибками (устойчивость). Большое C требует правильной классификации всего, зазор сужается (риск переобучения).

**Параметр gamma**: управляет «мягкостью» RBF-ядра. Малая gamma — широкое влияние каждой точки, гладкая граница. Большая gamma — узкое влияние, граница плотно огибает точки.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

pipe = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
grid = GridSearchCV(
    pipe,
    {'svc__C': [0.1, 1, 10, 100],
     'svc__gamma': ['scale', 0.01, 0.1, 1]},
    cv=5, n_jobs=-1)
grid.fit(X_train, y_train)
print('Лучшие параметры:', grid.best_params_)
print(f'Качество на кросс-валидации: {grid.best_score_:.3f}')

### Качество классификации

Оценим лучшую модель на отложенной тестовой выборке.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = grid.predict(X_test)
print(f'Доля верных ответов на тесте: {accuracy_score(y_test, y_pred):.3f}\n')
print(classification_report(y_test, y_pred, target_names=class_names))

### Визуализация разделяющей границы на синтетических данных

На двумерном наборе `make_moons` (два переплетающихся «полумесяца») сравним три ядра: линейное, полиномиальное и RBF. Это классический пример, где линейная граница заведомо не справится, и наглядно видно, что даёт нелинейность.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

Xs = StandardScaler().fit_transform(X2d)
kernels = [('Линейное ядро', SVC(kernel='linear', C=1)),
           ('Полиномиальное ядро', SVC(kernel='poly', degree=3, C=1)),
           ('RBF-ядро', SVC(kernel='rbf', C=10, gamma=1))]

h = 0.02
xx, yy = np.meshgrid(
    np.arange(Xs[:, 0].min() - 0.5, Xs[:, 0].max() + 0.5, h),
    np.arange(Xs[:, 1].min() - 0.5, Xs[:, 1].max() + 0.5, h))
cmap_bg = ListedColormap([VIZ['node_fill'], '#dfe9fb'])

fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.0))
for ax, (name, clf) in zip(axes, kernels):
    clf.fit(Xs, y2d)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=cmap_bg, alpha=0.8)
    for cls, color in zip([0, 1], [VIZ['series'][0],
                                   VIZ['series'][2]]):
        m = y2d == cls
        ax.scatter(Xs[m, 0], Xs[m, 1], color=color,
                   edgecolor='white', s=35,
                   label=f'класс {cls}')
    sv = clf.support_
    ax.scatter(Xs[sv, 0], Xs[sv, 1], s=130, facecolors='none',
               edgecolors=VIZ['series'][3], linewidths=1.6,
               label='опорные векторы')
    ax.set_title(name)
    ax.set_xlabel('Признак 1')
    ax.set_ylabel('Признак 2')
    ax.grid(False)
axes[0].legend(loc='upper left')
fig.tight_layout()
plt.show()

**Как читать эти графики.** Три панели — три разных ядра SVM на одних и тех же данных (два переплетающихся полумесяца). Цвет фона — предсказанный класс. Точки — реальные наблюдения. Обведённые кружками точки — опорные векторы.

- Линейное ядро: строит прямую границу; для этих данных она неприемлема — неизбежно ошибается на перекрывающейся части.
- Полиномиальное ядро: граница изогнута, справляется лучше, но ещё не идеально.
- RBF-ядро: граница огибает оба «полумесяца», разделяя их почти без ошибок. Заметьте, что опорных векторов теперь меньше — модель нашла компактное описание границы.

Число обведённых точек — косвенный показатель сложности задачи: много опорных векторов означает сложную или «грязную» (зашумлённую) границу.

## 5. Интерпретация результата

- **Лучшие параметры и качество**: решёточный поиск выбирает `C` и `gamma` по кросс-валидации; качество на отложенном тесте — несмещённая оценка обобщающей способности.
- **Разделяющая граница**: линейное ядро строит прямую границу и не справляется с переплетёнными классами; полиномиальное и RBF-ядро дают гибкие нелинейные границы.
- **Опорные векторы**: только обведённые наблюдения влияют на границу. Их малое число означает компактную, устойчивую модель; большое — что задача сложна или модель переобучена.
- **Влияние `C` и `gamma`**: большое `C` и большое `gamma` дают извилистую границу, плотно облегающую обучающие точки, — риск переобучения.

SVM не выдаёт вероятностей напрямую; при необходимости включите `probability=True` или используйте калибровку.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными для задачи классификации.

1. Загрузите файл в Colab через вкладку «Файлы».
2. Снимите комментарии в ячейке ниже и укажите имя файла и столбец с меткой класса.
3. Выполните ячейки раздела 4 (визуализация границы в разделе 4 работает только для двух признаков).

## 7. Поэкспериментируйте сами

1. **Влияние параметра C.** В ячейке линейного эксперимента (раздел «Наглядный эксперимент») задайте `C=0.01` и `C=100`. Как меняется ширина зазора и число опорных векторов? При малом C зазор широкий, но допускаются нарушения; при большом — узкий, но правило жёстче.

2. **Влияние шума.** Измените `noise=0.25` на `noise=0.45` при создании `make_moons`. Как изменится граница RBF-ядра? Станет ли она более извилистой? Это демонстрирует, как шум в данных влияет на сложность модели.

3. **Добавьте вероятности.** В `SVC` задайте `probability=True` и повторите обучение. Используйте `grid.best_estimator_.predict_proba(X_test)` для получения вероятностей классов. Почему по умолчанию SVM не выдаёт вероятности (в отличие от логистической регрессии)?

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_data.csv')
# target_column = 'класс'
#
# y = df[target_column]
# X = df.drop(columns=[target_column]).select_dtypes('number')
# class_names = [str(c) for c in sorted(y.unique())]
#
# print(f'Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}')
# Далее повторите ячейки раздела 4 (поиск параметров и оценку).

## 8. Краткие выводы

- SVM ищет разделяющую границу с максимальным зазором до ближайших наблюдений — это делает модель устойчивой к небольшим измерительным ошибкам и выбросам.
- Только опорные векторы (наблюдения на границах зазора) определяют решение; всё остальное не влияет.
- Ядровый приём позволяет строить нелинейные границы без явного вычисления признаков в многомерном пространстве. RBF-ядро — универсальный выбор по умолчанию.
- Параметры C и gamma существенно влияют на сложность границы: их нужно подбирать кросс-валидацией.
- SVM не выдаёт калиброванные вероятности напрямую; при необходимости используйте `probability=True` (медленно) или калибровку по Платту.
- Метод хорошо работает при небольшом числе наблюдений и высокоразмерных данных; при очень больших выборках предпочтительнее `LinearSVC` или логистическая регрессия.